In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/notebooks/fda137/spectral-graffiti-test/sample_submission.csv
/kaggle/input/notebooks/fda137/spectral-graffiti-test/test_ground_truth.csv
/kaggle/input/notebooks/fda137/spectral-graffiti-test/__results__.html
/kaggle/input/notebooks/fda137/spectral-graffiti-test/__notebook__.ipynb
/kaggle/input/notebooks/fda137/spectral-graffiti-test/__output__.json
/kaggle/input/notebooks/fda137/spectral-graffiti-test/test_features.csv
/kaggle/input/notebooks/fda137/spectral-graffiti-test/custom.css


In [2]:

import os
import gc
import math
import time
import copy
import random
import warnings
from dataclasses import dataclass
from itertools import product

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")


# ============================================================
# CONFIG
# ============================================================
@dataclass
class CFG:
    TRAIN_CSV: str = "/kaggle/input/datasets/nikhil75817581/dataset/spectral_graffiti_headley_recovered.csv"
    TEST_CSV: str = ""   # set if separate hidden test csv exists

    OUTPUT_DIR: str = "/kaggle/working/round1_3fold_outputs"
    SEARCH_RESULTS_CSV: str = "search_results_3fold.csv"
    FINAL_MODEL_NAME: str = "final_best_model_3fold.pt"
    SUBMISSION_NAME: str = "submission_targets_only.csv"

    seed: int = 42
    seq_len: int = 100
    num_folds: int = 3

    num_workers: int = 2
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    use_amp: bool = True

    # Search controls
    max_trials: int = 6         # increase if you want more
    tune_subset: int | None = 30000   # None => use all 80k for search
    final_epochs_cap: int = 60

    # Logging
    print_every: int = 5


cfg = CFG()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)


# ============================================================
# REPRODUCIBILITY
# ============================================================
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(cfg.seed)


# ============================================================
# UTILS
# ============================================================
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def normalize_time_grid(times: np.ndarray) -> np.ndarray:
    tmin = times.min(axis=1, keepdims=True)
    tmax = times.max(axis=1, keepdims=True)
    return (times - tmin) / np.maximum(tmax - tmin, 1e-6)


def fit_value_normalization(train_values: np.ndarray, train_context_mask: np.ndarray):
    observed_vals = train_values[train_context_mask > 0.5]
    mean = float(observed_vals.mean())
    std = float(observed_vals.std() + 1e-6)
    return mean, std


def apply_value_normalization(values: np.ndarray, mean: float, std: float):
    return (values - mean) / std


def masked_mse(pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor, eps: float = 1e-8):
    loss = ((pred - target) ** 2) * mask
    denom = mask.sum().clamp_min(eps)
    return loss.sum() / denom


class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0, mode="min"):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best = None
        self.counter = 0
        self.stop = False

    def step(self, value):
        if self.best is None:
            self.best = value
            return True

        if self.mode == "min":
            improved = value < (self.best - self.min_delta)
        else:
            improved = value > (self.best + self.min_delta)

        if improved:
            self.best = value
            self.counter = 0
            return True

        self.counter += 1
        if self.counter >= self.patience:
            self.stop = True
        return False


def cosine_scheduler(optimizer, warmup_steps, total_steps):
    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / max(1, warmup_steps)
        progress = float(step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# ============================================================
# ROBUST COLUMN CLEANING
# ============================================================
def clean_column_name(col: str) -> str:
    col = str(col)
    col = col.replace("\ufeff", "")
    col = col.replace("\u200b", "")
    col = col.strip()

    if col.endswith("Sample_ID"):
        return "Sample_ID"
    if col.endswith("Time_ms"):
        return "Time_ms"
    if col.endswith("Is_Context"):
        return "Is_Context"
    if col.endswith("Value"):
        return "Value"
    return col


# ============================================================
# LOAD DATA
# ============================================================
def load_signal_csv(csv_path: str, seq_len: int = 100):
    print(f"Loading CSV: {csv_path}")
    df = pd.read_csv(csv_path)

    original_cols = list(df.columns)
    df.columns = [clean_column_name(c) for c in df.columns]
    cleaned_cols = list(df.columns)

    print("Original columns:", original_cols)
    print("Cleaned columns :", cleaned_cols)

    required = {"Sample_ID", "Time_ms", "Is_Context", "Value"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns after cleaning: {missing}")

    df = df[["Sample_ID", "Time_ms", "Is_Context", "Value"]].copy()

    df["Sample_ID"] = pd.to_numeric(df["Sample_ID"], errors="coerce")
    df["Time_ms"] = pd.to_numeric(df["Time_ms"], errors="coerce")
    df["Is_Context"] = pd.to_numeric(df["Is_Context"], errors="coerce")
    df["Value"] = pd.to_numeric(df["Value"], errors="coerce")

    if df[["Sample_ID", "Time_ms", "Is_Context", "Value"]].isnull().any().any():
        raise ValueError("Found NaNs after numeric conversion. Please check CSV contents.")

    df = df.sort_values(["Sample_ID", "Time_ms"]).reset_index(drop=True)

    sample_ids_all = df["Sample_ID"].to_numpy(dtype=np.int64)
    time_ms_all = df["Time_ms"].to_numpy(dtype=np.float32)
    context_all = df["Is_Context"].to_numpy(dtype=np.float32)
    value_all = df["Value"].to_numpy(dtype=np.float32)

    uniq_ids, counts = np.unique(sample_ids_all, return_counts=True)
    if not np.all(counts == seq_len):
        raise ValueError(
            f"Every Sample_ID must have exactly {seq_len} rows. "
            f"Found min={counts.min()}, max={counts.max()}"
        )

    n_samples = len(uniq_ids)
    times = time_ms_all.reshape(n_samples, seq_len)
    context_mask = context_all.reshape(n_samples, seq_len)
    values = value_all.reshape(n_samples, seq_len)

    print(f"Loaded {n_samples:,} samples")
    print(f"Observed/context fraction: {context_mask.mean():.4f}")
    return uniq_ids, times, context_mask, values


# ============================================================
# DATASETS
# ============================================================
class SupervisedDataset(Dataset):
    """
    visible input = Value at Is_Context == 1
    target loss   = Value at Is_Context == 0
    """
    def __init__(self, times, context_mask, values, mean, std):
        self.x = normalize_time_grid(times).astype(np.float32)
        self.context_mask = context_mask.astype(np.float32)
        self.values = apply_value_normalization(values.astype(np.float32), mean, std)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        x = self.x[idx]
        c = self.context_mask[idx]
        y = self.values[idx]

        observed_input_values = y * c
        target_mask = (1.0 - c).astype(np.float32)

        return {
            "x": torch.tensor(x, dtype=torch.float32),
            "y": torch.tensor(y, dtype=torch.float32),
            "visible_mask": torch.tensor(c, dtype=torch.float32),
            "target_mask": torch.tensor(target_mask, dtype=torch.float32),
            "observed_input_values": torch.tensor(observed_input_values, dtype=torch.float32),
        }


class InferenceDataset(Dataset):
    def __init__(self, sample_ids, times, context_mask, values, mean, std):
        self.sample_ids = sample_ids
        self.raw_times = times.astype(np.float32)
        self.x = normalize_time_grid(times).astype(np.float32)
        self.context_mask = context_mask.astype(np.float32)
        self.values = apply_value_normalization(values.astype(np.float32), mean, std)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        x = self.x[idx]
        c = self.context_mask[idx]
        y = self.values[idx]
        observed_input_values = y * c
        target_mask = (1.0 - c).astype(np.float32)

        return {
            "sample_id": torch.tensor(self.sample_ids[idx], dtype=torch.long),
            "time_ms": torch.tensor(self.raw_times[idx], dtype=torch.float32),
            "x": torch.tensor(x, dtype=torch.float32),
            "visible_mask": torch.tensor(c, dtype=torch.float32),
            "target_mask": torch.tensor(target_mask, dtype=torch.float32),
            "observed_input_values": torch.tensor(observed_input_values, dtype=torch.float32),
        }


# ============================================================
# MODEL
# ============================================================
def sinusoidal_features(x, num_bands=16):
    device = x.device
    freqs = 2.0 ** torch.arange(num_bands, device=device, dtype=torch.float32) * math.pi
    x_exp = x.unsqueeze(-1)
    angles = x_exp * freqs.view(1, 1, -1)
    return torch.cat([x_exp, torch.sin(angles), torch.cos(angles)], dim=-1)


class ConvResidualBlock(nn.Module):
    def __init__(self, channels, dilation=1, dropout=0.1):
        super().__init__()
        self.bn1 = nn.BatchNorm1d(channels)
        self.conv1 = nn.Conv1d(channels, channels, kernel_size=3, padding=dilation, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(channels)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size=3, padding=dilation, dilation=dilation)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        residual = x
        x = self.conv1(F.gelu(self.bn1(x)))
        x = self.dropout(x)
        x = self.conv2(F.gelu(self.bn2(x)))
        x = self.dropout(x)
        return x + residual


class HybridConvEncoder(nn.Module):
    def __init__(self, input_dim, d_model, num_blocks=4, dropout=0.1):
        super().__init__()
        self.in_proj = nn.Conv1d(input_dim, d_model, kernel_size=1)
        self.blocks = nn.ModuleList([
            ConvResidualBlock(d_model, dilation=(2 ** i), dropout=dropout)
            for i in range(num_blocks)
        ])
        self.out_norm = nn.LayerNorm(d_model)

    def forward(self, feats):
        x = feats.transpose(1, 2)
        x = self.in_proj(x)
        for block in self.blocks:
            x = block(x)
        x = x.transpose(1, 2)
        return self.out_norm(x)


class CrossAttentionBlock(nn.Module):
    def __init__(self, d_model, num_heads, ff_mult=4, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * ff_mult),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * ff_mult, d_model),
            nn.Dropout(dropout),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, q, memory, memory_key_padding_mask=None):
        q2, _ = self.self_attn(q, q, q, need_weights=False)
        q = self.norm1(q + q2)

        q2, _ = self.cross_attn(
            q, memory, memory,
            key_padding_mask=memory_key_padding_mask,
            need_weights=False
        )
        q = self.norm2(q + q2)

        q2 = self.ff(q)
        q = self.norm3(q + q2)
        return q


class CrossAttentionDecoder(nn.Module):
    def __init__(self, time_dim, d_model, num_heads, num_layers, ff_mult=4, dropout=0.1):
        super().__init__()
        self.query_proj = nn.Sequential(
            nn.Linear(time_dim, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )
        self.layers = nn.ModuleList([
            CrossAttentionBlock(d_model, num_heads, ff_mult=ff_mult, dropout=dropout)
            for _ in range(num_layers)
        ])
        self.out_norm = nn.LayerNorm(d_model)

    def forward(self, query_time_features, memory, memory_key_padding_mask=None):
        q = self.query_proj(query_time_features)
        for layer in self.layers:
            q = layer(q, memory, memory_key_padding_mask=memory_key_padding_mask)
        return self.out_norm(q)


class HybridInpaintingModel(nn.Module):
    def __init__(self, hp):
        super().__init__()

        if hp["d_model"] % hp["num_heads"] != 0:
            raise ValueError(
                f"Invalid config: d_model={hp['d_model']} must be divisible by num_heads={hp['num_heads']}"
            )

        self.num_bands = hp["num_bands"]
        time_dim = 1 + 2 * self.num_bands
        input_dim = 2 + time_dim

        self.encoder = HybridConvEncoder(
            input_dim=input_dim,
            d_model=hp["d_model"],
            num_blocks=hp["num_conv_blocks"],
            dropout=hp["dropout"],
        )

        self.decoder = CrossAttentionDecoder(
            time_dim=time_dim,
            d_model=hp["d_model"],
            num_heads=hp["num_heads"],
            num_layers=hp["num_decoder_layers"],
            ff_mult=hp["ff_mult"],
            dropout=hp["dropout"],
        )

        self.head = nn.Sequential(
            nn.Linear(hp["d_model"], hp["d_model"]),
            nn.GELU(),
            nn.Dropout(hp["dropout"]),
            nn.Linear(hp["d_model"], 1),
        )

    def forward(self, x, observed_input_values, visible_mask):
        time_feats = sinusoidal_features(x, num_bands=self.num_bands)
        enc_feats = torch.cat([
            observed_input_values.unsqueeze(-1),
            visible_mask.unsqueeze(-1),
            time_feats
        ], dim=-1)

        memory = self.encoder(enc_feats)
        memory_key_padding_mask = (visible_mask < 0.5)

        decoded = self.decoder(
            query_time_features=time_feats,
            memory=memory,
            memory_key_padding_mask=memory_key_padding_mask
        )

        pred = self.head(decoded).squeeze(-1)
        return pred


# ============================================================
# TRAIN / VALID / INFERENCE
# ============================================================
def train_one_epoch(model, loader, optimizer, scheduler, scaler, device, use_amp=True, grad_clip=1.0):
    model.train()
    total_loss = 0.0
    total_n = 0

    for batch in loader:
        x = batch["x"].to(device)
        y = batch["y"].to(device)
        visible_mask = batch["visible_mask"].to(device)
        target_mask = batch["target_mask"].to(device)
        observed_input_values = batch["observed_input_values"].to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(use_amp and device.startswith("cuda"))):
            pred = model(x, observed_input_values, visible_mask)
            loss = masked_mse(pred, y, target_mask)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        bs = x.size(0)
        total_loss += loss.item() * bs
        total_n += bs

    return total_loss / max(1, total_n)


@torch.no_grad()
def valid_one_epoch(model, loader, device):
    model.eval()
    total_loss = 0.0
    total_n = 0

    for batch in loader:
        x = batch["x"].to(device)
        y = batch["y"].to(device)
        visible_mask = batch["visible_mask"].to(device)
        target_mask = batch["target_mask"].to(device)
        observed_input_values = batch["observed_input_values"].to(device)

        pred = model(x, observed_input_values, visible_mask)
        loss = masked_mse(pred, y, target_mask)

        bs = x.size(0)
        total_loss += loss.item() * bs
        total_n += bs

    return total_loss / max(1, total_n)


@torch.no_grad()
def predict_dataset(model, loader, device, mean, std):
    model.eval()

    all_sample_ids = []
    all_time_ms = []
    all_target_masks = []
    all_preds = []

    for batch in loader:
        sample_id = batch["sample_id"].cpu().numpy()       # [B]
        time_ms = batch["time_ms"].cpu().numpy()           # [B, L]
        x = batch["x"].to(device)
        visible_mask = batch["visible_mask"].to(device)
        observed_input_values = batch["observed_input_values"].to(device)
        target_mask = batch["target_mask"].cpu().numpy()

        pred = model(x, observed_input_values, visible_mask)
        pred = pred.cpu().numpy() * std + mean

        all_sample_ids.append(sample_id)
        all_time_ms.append(time_ms)
        all_target_masks.append(target_mask)
        all_preds.append(pred)

    all_sample_ids = np.concatenate(all_sample_ids, axis=0)
    all_time_ms = np.concatenate(all_time_ms, axis=0)
    all_target_masks = np.concatenate(all_target_masks, axis=0)
    all_preds = np.concatenate(all_preds, axis=0)

    return all_sample_ids, all_time_ms, all_target_masks, all_preds


# ============================================================
# SEARCH SPACE
# ============================================================
search_space = {
    "d_model": [128, 160, 192, 256],
    "num_conv_blocks": [3, 4, 5],
    "num_heads": [4, 5, 6, 8],
    "num_decoder_layers": [3, 4, 5],
    "ff_mult": [4],
    "num_bands": [16],
    "dropout": [0.05, 0.10, 0.15],
    "lr": [1.5e-4, 2e-4, 3e-4],
    "weight_decay": [1e-4, 3e-4, 5e-4],
    "batch_size": [128],
    "epochs": [80],
    "early_stop_patience": [5, 7],
    "grad_clip": [1.0],
}


def build_trial_list(space, max_trials=None, seed=42):
    keys = list(space.keys())
    values = [space[k] for k in keys]
    raw_trials = [dict(zip(keys, combo)) for combo in product(*values)]

    valid_trials = []
    for hp in raw_trials:
        if hp["d_model"] % hp["num_heads"] == 0:
            valid_trials.append(hp)

    random.Random(seed).shuffle(valid_trials)

    if max_trials is not None:
        valid_trials = valid_trials[:max_trials]

    return valid_trials


# ============================================================
# LOAD MAIN DATA
# ============================================================
sample_ids, times, context_mask, values = load_signal_csv(cfg.TRAIN_CSV, seq_len=cfg.seq_len)

all_indices = np.arange(len(sample_ids))
if cfg.tune_subset is not None and cfg.tune_subset < len(all_indices):
    rng = np.random.RandomState(cfg.seed)
    search_indices = rng.choice(all_indices, size=cfg.tune_subset, replace=False)
    search_indices = np.sort(search_indices)
else:
    search_indices = all_indices

search_sample_ids = sample_ids[search_indices]
search_times = times[search_indices]
search_context = context_mask[search_indices]
search_values = values[search_indices]

print(f"\nUsing {len(search_sample_ids):,} samples for 3-fold hyperparameter search")


# ============================================================
# SINGLE TRIAL ACROSS 3 FOLDS
# ============================================================
def run_one_trial_3fold(hp, sample_ids, times, context_mask, values, cfg, trial_seed=42):
    fold_scores = []
    fold_best_epochs = []
    fold_rows = []

    kf = KFold(n_splits=cfg.num_folds, shuffle=True, random_state=trial_seed)

    for fold, (train_idx, val_idx) in enumerate(kf.split(sample_ids), start=1):
        print(f"\n  Fold {fold}/{cfg.num_folds}")

        train_times = times[train_idx]
        train_context = context_mask[train_idx]
        train_values = values[train_idx]

        val_times = times[val_idx]
        val_context = context_mask[val_idx]
        val_values = values[val_idx]

        fold_mean, fold_std = fit_value_normalization(train_values, train_context)

        train_ds = SupervisedDataset(train_times, train_context, train_values, fold_mean, fold_std)
        val_ds = SupervisedDataset(val_times, val_context, val_values, fold_mean, fold_std)

        train_loader = DataLoader(
            train_ds,
            batch_size=hp["batch_size"],
            shuffle=True,
            num_workers=cfg.num_workers,
            pin_memory=True,
            drop_last=False,
        )
        val_loader = DataLoader(
            val_ds,
            batch_size=hp["batch_size"],
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=True,
            drop_last=False,
        )

        model = HybridInpaintingModel(hp).to(cfg.device)
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=hp["lr"],
            weight_decay=hp["weight_decay"]
        )

        steps_per_epoch = len(train_loader)
        total_steps = hp["epochs"] * max(1, steps_per_epoch)
        warmup_steps = int(0.08 * total_steps)
        scheduler = cosine_scheduler(optimizer, warmup_steps, total_steps)
        scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and cfg.device.startswith("cuda")))
        early_stopper = EarlyStopping(
            patience=hp["early_stop_patience"],
            min_delta=1e-5,
            mode="min"
        )

        best_val = float("inf")
        best_epoch = -1

        for epoch in range(1, hp["epochs"] + 1):
            train_loss = train_one_epoch(
                model, train_loader, optimizer, scheduler, scaler,
                cfg.device, use_amp=cfg.use_amp, grad_clip=hp["grad_clip"]
            )
            val_loss = valid_one_epoch(model, val_loader, cfg.device)

            if epoch == 1 or epoch % cfg.print_every == 0:
                print(f"    Epoch {epoch:02d}/{hp['epochs']} | train={train_loss:.6f} | val={val_loss:.6f}")

            improved = early_stopper.step(val_loss)
            if improved:
                best_val = val_loss
                best_epoch = epoch

            if early_stopper.stop:
                print(f"    Early stopped at epoch {epoch}")
                break

        fold_scores.append(best_val)
        fold_best_epochs.append(best_epoch)
        fold_rows.append({
            "fold": fold,
            "fold_best_val": best_val,
            "fold_best_epoch": best_epoch
        })

        del model, optimizer, scheduler, scaler, train_loader, val_loader, train_ds, val_ds
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    mean_score = float(np.mean(fold_scores))
    std_score = float(np.std(fold_scores))
    mean_epoch = int(round(np.mean(fold_best_epochs)))

    return {
        "mean_val_masked_mse": mean_score,
        "std_val_masked_mse": std_score,
        "mean_best_epoch": mean_epoch,
        "fold_rows": fold_rows,
    }


# ============================================================
# HYPERPARAMETER SEARCH WITH 3-FOLD VALIDATION
# ============================================================
trials = build_trial_list(search_space, max_trials=cfg.max_trials, seed=cfg.seed)
print(f"\nRunning {len(trials)} valid trials with {cfg.num_folds}-fold validation")

search_rows = []
best_trial = None
best_val = float("inf")

for trial_id, hp in enumerate(trials, start=1):
    print("\n" + "=" * 90)
    print(f"Trial {trial_id}/{len(trials)}")
    print(hp)

    seed_everything(cfg.seed + trial_id)

    try:
        t0 = time.time()
        out = run_one_trial_3fold(
            hp=hp,
            sample_ids=search_sample_ids,
            times=search_times,
            context_mask=search_context,
            values=search_values,
            cfg=cfg,
            trial_seed=cfg.seed + trial_id
        )
        elapsed = time.time() - t0

        row = copy.deepcopy(hp)
        row["trial_id"] = trial_id
        row["mean_val_masked_mse"] = out["mean_val_masked_mse"]
        row["std_val_masked_mse"] = out["std_val_masked_mse"]
        row["mean_best_epoch"] = out["mean_best_epoch"]
        row["status"] = "ok"
        row["elapsed_sec"] = elapsed
        search_rows.append(row)

        print(f"\n  Trial mean val masked MSE: {out['mean_val_masked_mse']:.6f} ± {out['std_val_masked_mse']:.6f}")
        print(f"  Trial mean best epoch: {out['mean_best_epoch']}")

        if out["mean_val_masked_mse"] < best_val:
            best_val = out["mean_val_masked_mse"]
            best_trial = copy.deepcopy(hp)
            best_trial["mean_val_masked_mse"] = out["mean_val_masked_mse"]
            best_trial["std_val_masked_mse"] = out["std_val_masked_mse"]
            best_trial["mean_best_epoch"] = out["mean_best_epoch"]

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except RuntimeError as e:
        row = copy.deepcopy(hp)
        row["trial_id"] = trial_id
        row["mean_val_masked_mse"] = np.nan
        row["std_val_masked_mse"] = np.nan
        row["mean_best_epoch"] = np.nan
        row["status"] = f"runtime_error: {str(e)[:200]}"
        row["elapsed_sec"] = np.nan
        search_rows.append(row)
        print(f"  Skipping failed trial due to runtime error: {e}")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

if best_trial is None:
    raise RuntimeError("All trials failed. Reduce model size or search space.")

search_df = pd.DataFrame(search_rows)
search_df = search_df.sort_values(
    by=["status", "mean_val_masked_mse"],
    ascending=[True, True],
    na_position="last"
).reset_index(drop=True)

search_csv_path = os.path.join(cfg.OUTPUT_DIR, cfg.SEARCH_RESULTS_CSV)
search_df.to_csv(search_csv_path, index=False)

print("\n" + "=" * 90)
print("3-fold search complete.")
print("Best mean validation masked MSE:", best_val)
print("Best config:")
print(best_trial)
print(f"Search results saved to: {search_csv_path}")


# ============================================================
# RETRAIN FRESH MODEL ON FULL TRAINING DATA
# ============================================================
print("\n" + "=" * 90)
print("Retraining fresh model on full training data with best config")
print("=" * 90)

best_hp = copy.deepcopy(best_trial)

full_mean, full_std = fit_value_normalization(values, context_mask)
seed_everything(cfg.seed + 999)

final_train_ds = SupervisedDataset(times, context_mask, values, full_mean, full_std)
final_train_loader = DataLoader(
    final_train_ds,
    batch_size=best_hp["batch_size"],
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=True,
    drop_last=False,
)

final_model = HybridInpaintingModel(best_hp).to(cfg.device)
final_optimizer = torch.optim.AdamW(
    final_model.parameters(),
    lr=best_hp["lr"],
    weight_decay=best_hp["weight_decay"]
)

final_epochs = min(max(1, int(best_hp["mean_best_epoch"]) + 2), cfg.final_epochs_cap)
final_total_steps = final_epochs * max(1, len(final_train_loader))
final_warmup_steps = int(0.08 * final_total_steps)
final_scheduler = cosine_scheduler(final_optimizer, final_warmup_steps, final_total_steps)
final_scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and cfg.device.startswith("cuda")))

print(f"Final retrain epochs: {final_epochs}")

for epoch in range(1, final_epochs + 1):
    train_loss = train_one_epoch(
        final_model,
        final_train_loader,
        final_optimizer,
        final_scheduler,
        final_scaler,
        cfg.device,
        use_amp=cfg.use_amp,
        grad_clip=best_hp["grad_clip"]
    )
    if epoch == 1 or epoch % cfg.print_every == 0 or epoch == final_epochs:
        print(f"  Retrain epoch {epoch:02d}/{final_epochs} | train={train_loss:.6f}")

final_model_path = os.path.join(cfg.OUTPUT_DIR, cfg.FINAL_MODEL_NAME)
torch.save(
    {
        "model_state_dict": final_model.state_dict(),
        "hp": best_hp,
        "value_mean": full_mean,
        "value_std": full_std,
    },
    final_model_path
)
print(f"Saved final retrained model to: {final_model_path}")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ============================================================
# INFERENCE / SUBMISSION
# Only target rows
# ============================================================
inference_csv = cfg.TEST_CSV if str(cfg.TEST_CSV).strip() != "" else cfg.TRAIN_CSV
print(f"\nInference CSV: {inference_csv}")

inf_sample_ids, inf_times, inf_context_mask, inf_values = load_signal_csv(
    inference_csv,
    seq_len=cfg.seq_len
)

ckpt = torch.load(final_model_path, map_location=cfg.device)
final_model.load_state_dict(ckpt["model_state_dict"])
full_mean = float(ckpt["value_mean"])
full_std = float(ckpt["value_std"])

inf_ds = InferenceDataset(
    sample_ids=inf_sample_ids,
    times=inf_times,
    context_mask=inf_context_mask,
    values=inf_values,
    mean=full_mean,
    std=full_std,
)
inf_loader = DataLoader(
    inf_ds,
    batch_size=best_hp["batch_size"],
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True,
    drop_last=False,
)

pred_sample_ids, pred_time_ms, pred_target_masks, pred_values = predict_dataset(
    final_model, inf_loader, cfg.device, full_mean, full_std
)

print("Shapes before flatten:")
print("pred_sample_ids   :", pred_sample_ids.shape)
print("pred_time_ms      :", pred_time_ms.shape)
print("pred_target_masks :", pred_target_masks.shape)
print("pred_values       :", pred_values.shape)

n_samples, seq_len = pred_values.shape

flat_sample_ids = np.repeat(pred_sample_ids, seq_len)
flat_time_ms = pred_time_ms.reshape(-1)
flat_target_masks = pred_target_masks.reshape(-1)
flat_pred_values = pred_values.reshape(-1)

print("\nShapes after flatten:")
print("flat_sample_ids   :", flat_sample_ids.shape)
print("flat_time_ms      :", flat_time_ms.shape)
print("flat_target_masks :", flat_target_masks.shape)
print("flat_pred_values  :", flat_pred_values.shape)

assert len(flat_sample_ids) == len(flat_time_ms) == len(flat_target_masks) == len(flat_pred_values), \
    "Flattened arrays do not have the same length."

submission = pd.DataFrame({
    "Sample_ID": flat_sample_ids.astype(np.int64),
    "Time_ms": flat_time_ms.astype(np.int64),
    "Predicted_Value": flat_pred_values.astype(np.float32),
})

submission = submission[flat_target_masks > 0.5].copy().reset_index(drop=True)

submission_path = os.path.join(cfg.OUTPUT_DIR, cfg.SUBMISSION_NAME)
submission.to_csv(submission_path, index=False)

print(f"\nSaved target-only submission to: {submission_path}")
print("Submission shape:", submission.shape)
print(submission.head(10))

Loading CSV: /kaggle/input/datasets/nikhil75817581/dataset/spectral_graffiti_headley_recovered.csv
Original columns: ['Sample_ID', 'Time_ms', 'Is_Context', 'Value']
Cleaned columns : ['Sample_ID', 'Time_ms', 'Is_Context', 'Value']
Loaded 80,000 samples
Observed/context fraction: 0.2000

Using 30,000 samples for 3-fold hyperparameter search

Running 6 valid trials with 3-fold validation

Trial 1/6
{'d_model': 192, 'num_conv_blocks': 5, 'num_heads': 4, 'num_decoder_layers': 5, 'ff_mult': 4, 'num_bands': 16, 'dropout': 0.15, 'lr': 0.0003, 'weight_decay': 0.0001, 'batch_size': 128, 'epochs': 80, 'early_stop_patience': 5, 'grad_clip': 1.0}

  Fold 1/3
    Epoch 01/80 | train=1.008600 | val=0.997441
    Epoch 05/80 | train=0.067073 | val=0.044015
    Epoch 10/80 | train=0.046280 | val=0.037761
    Epoch 15/80 | train=0.041082 | val=0.035692
    Epoch 20/80 | train=0.037679 | val=0.034059
    Epoch 25/80 | train=0.035414 | val=0.030069
    Epoch 30/80 | train=0.033990 | val=0.032855
    Early

# Renamed submission_targets_only.csv as submission.csv

In [ ]:
# ============================================================
# ROUND 2 INFERENCE ONLY
# Load saved final model -> predict test.csv -> save submission
# ============================================================

import os
import math
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")


# ============================================================
# CONFIG
# ============================================================
@dataclass
class CFG:
    TEST_CSV: str = "test_features.csv"
    MODEL_PATH: str = "final_best_model_3fold.pt"

    OUTPUT_DIR: str = "Solution"
    SUBMISSION_NAME: str = "submission_Test"

    seq_len: int = 100
    batch_size: int = 128
    num_workers: int = 2
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

cfg = CFG()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)


# ============================================================
# ROBUST COLUMN CLEANING
# ============================================================
def clean_column_name(col: str) -> str:
    col = str(col)
    col = col.replace("\ufeff", "")
    col = col.replace("\u200b", "")
    col = col.strip()

    if col.endswith("Sample_ID"):
        return "Sample_ID"
    if col.endswith("Time_ms"):
        return "Time_ms"
    if col.endswith("Is_Context"):
        return "Is_Context"
    if col.endswith("Value"):
        return "Value"
    return col


# ============================================================
# LOAD CSV
# ============================================================
def load_signal_csv(csv_path: str, seq_len: int = 100):
    print(f"Loading CSV: {csv_path}")
    df = pd.read_csv(csv_path)

    original_cols = list(df.columns)
    df.columns = [clean_column_name(c) for c in df.columns]
    cleaned_cols = list(df.columns)

    print("Original columns:", original_cols)
    print("Cleaned columns :", cleaned_cols)

    required = {"Sample_ID", "Time_ms", "Is_Context", "Value"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns after cleaning: {missing}")

    df = df[["Sample_ID", "Time_ms", "Is_Context", "Value"]].copy()

    df["Sample_ID"] = pd.to_numeric(df["Sample_ID"], errors="coerce")
    df["Time_ms"] = pd.to_numeric(df["Time_ms"], errors="coerce")
    df["Is_Context"] = pd.to_numeric(df["Is_Context"], errors="coerce")
    df["Value"] = pd.to_numeric(df["Value"], errors="coerce")

    if df[["Sample_ID", "Time_ms", "Is_Context", "Value"]].isnull().any().any():
        raise ValueError("Found NaNs after numeric conversion. Please check CSV contents.")

    df = df.sort_values(["Sample_ID", "Time_ms"]).reset_index(drop=True)

    sample_ids_all = df["Sample_ID"].to_numpy(dtype=np.int64)
    time_ms_all = df["Time_ms"].to_numpy(dtype=np.float32)
    context_all = df["Is_Context"].to_numpy(dtype=np.float32)
    value_all = df["Value"].to_numpy(dtype=np.float32)

    uniq_ids, counts = np.unique(sample_ids_all, return_counts=True)
    if not np.all(counts == seq_len):
        raise ValueError(
            f"Every Sample_ID must have exactly {seq_len} rows. "
            f"Found min={counts.min()}, max={counts.max()}"
        )

    n_samples = len(uniq_ids)
    times = time_ms_all.reshape(n_samples, seq_len)
    context_mask = context_all.reshape(n_samples, seq_len)
    values = value_all.reshape(n_samples, seq_len)

    print(f"Loaded {n_samples:,} samples")
    print(f"Observed/context fraction: {context_mask.mean():.4f}")
    return uniq_ids, times, context_mask, values


# ============================================================
# NORMALIZATION
# ============================================================
def normalize_time_grid(times: np.ndarray) -> np.ndarray:
    tmin = times.min(axis=1, keepdims=True)
    tmax = times.max(axis=1, keepdims=True)
    return (times - tmin) / np.maximum(tmax - tmin, 1e-6)


def apply_value_normalization(values: np.ndarray, mean: float, std: float):
    return (values - mean) / std


# ============================================================
# DATASET
# ============================================================
class InferenceDataset(Dataset):
    def __init__(self, sample_ids, times, context_mask, values, mean, std):
        self.sample_ids = sample_ids
        self.raw_times = times.astype(np.float32)
        self.x = normalize_time_grid(times).astype(np.float32)
        self.context_mask = context_mask.astype(np.float32)
        self.values = apply_value_normalization(values.astype(np.float32), mean, std)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        x = self.x[idx]
        c = self.context_mask[idx]
        y = self.values[idx]

        observed_input_values = y * c
        target_mask = (1.0 - c).astype(np.float32)

        return {
            "sample_id": torch.tensor(self.sample_ids[idx], dtype=torch.long),
            "time_ms": torch.tensor(self.raw_times[idx], dtype=torch.float32),
            "x": torch.tensor(x, dtype=torch.float32),
            "visible_mask": torch.tensor(c, dtype=torch.float32),
            "target_mask": torch.tensor(target_mask, dtype=torch.float32),
            "observed_input_values": torch.tensor(observed_input_values, dtype=torch.float32),
        }


# ============================================================
# MODEL
# ============================================================
def sinusoidal_features(x, num_bands=16):
    device = x.device
    freqs = 2.0 ** torch.arange(num_bands, device=device, dtype=torch.float32) * math.pi
    x_exp = x.unsqueeze(-1)
    angles = x_exp * freqs.view(1, 1, -1)
    return torch.cat([x_exp, torch.sin(angles), torch.cos(angles)], dim=-1)


class ConvResidualBlock(nn.Module):
    def __init__(self, channels, dilation=1, dropout=0.1):
        super().__init__()
        self.bn1 = nn.BatchNorm1d(channels)
        self.conv1 = nn.Conv1d(channels, channels, kernel_size=3, padding=dilation, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(channels)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size=3, padding=dilation, dilation=dilation)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        residual = x
        x = self.conv1(F.gelu(self.bn1(x)))
        x = self.dropout(x)
        x = self.conv2(F.gelu(self.bn2(x)))
        x = self.dropout(x)
        return x + residual


class HybridConvEncoder(nn.Module):
    def __init__(self, input_dim, d_model, num_blocks=4, dropout=0.1):
        super().__init__()
        self.in_proj = nn.Conv1d(input_dim, d_model, kernel_size=1)
        self.blocks = nn.ModuleList([
            ConvResidualBlock(d_model, dilation=(2 ** i), dropout=dropout)
            for i in range(num_blocks)
        ])
        self.out_norm = nn.LayerNorm(d_model)

    def forward(self, feats):
        x = feats.transpose(1, 2)
        x = self.in_proj(x)
        for block in self.blocks:
            x = block(x)
        x = x.transpose(1, 2)
        return self.out_norm(x)


class CrossAttentionBlock(nn.Module):
    def __init__(self, d_model, num_heads, ff_mult=4, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * ff_mult),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * ff_mult, d_model),
            nn.Dropout(dropout),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, q, memory, memory_key_padding_mask=None):
        q2, _ = self.self_attn(q, q, q, need_weights=False)
        q = self.norm1(q + q2)

        q2, _ = self.cross_attn(
            q, memory, memory,
            key_padding_mask=memory_key_padding_mask,
            need_weights=False
        )
        q = self.norm2(q + q2)

        q2 = self.ff(q)
        q = self.norm3(q + q2)
        return q


class CrossAttentionDecoder(nn.Module):
    def __init__(self, time_dim, d_model, num_heads, num_layers, ff_mult=4, dropout=0.1):
        super().__init__()
        self.query_proj = nn.Sequential(
            nn.Linear(time_dim, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )
        self.layers = nn.ModuleList([
            CrossAttentionBlock(d_model, num_heads, ff_mult=ff_mult, dropout=dropout)
            for _ in range(num_layers)
        ])
        self.out_norm = nn.LayerNorm(d_model)

    def forward(self, query_time_features, memory, memory_key_padding_mask=None):
        q = self.query_proj(query_time_features)
        for layer in self.layers:
            q = layer(q, memory, memory_key_padding_mask=memory_key_padding_mask)
        return self.out_norm(q)


class HybridInpaintingModel(nn.Module):
    def __init__(self, hp):
        super().__init__()

        if hp["d_model"] % hp["num_heads"] != 0:
            raise ValueError(
                f"Invalid config: d_model={hp['d_model']} must be divisible by num_heads={hp['num_heads']}"
            )

        self.num_bands = hp["num_bands"]
        time_dim = 1 + 2 * self.num_bands
        input_dim = 2 + time_dim

        self.encoder = HybridConvEncoder(
            input_dim=input_dim,
            d_model=hp["d_model"],
            num_blocks=hp["num_conv_blocks"],
            dropout=hp["dropout"],
        )

        self.decoder = CrossAttentionDecoder(
            time_dim=time_dim,
            d_model=hp["d_model"],
            num_heads=hp["num_heads"],
            num_layers=hp["num_decoder_layers"],
            ff_mult=hp["ff_mult"],
            dropout=hp["dropout"],
        )

        self.head = nn.Sequential(
            nn.Linear(hp["d_model"], hp["d_model"]),
            nn.GELU(),
            nn.Dropout(hp["dropout"]),
            nn.Linear(hp["d_model"], 1),
        )

    def forward(self, x, observed_input_values, visible_mask):
        time_feats = sinusoidal_features(x, num_bands=self.num_bands)
        enc_feats = torch.cat([
            observed_input_values.unsqueeze(-1),
            visible_mask.unsqueeze(-1),
            time_feats
        ], dim=-1)

        memory = self.encoder(enc_feats)
        memory_key_padding_mask = (visible_mask < 0.5)

        decoded = self.decoder(
            query_time_features=time_feats,
            memory=memory,
            memory_key_padding_mask=memory_key_padding_mask
        )

        pred = self.head(decoded).squeeze(-1)
        return pred


# ============================================================
# PREDICT
# ============================================================
@torch.no_grad()
def predict_dataset(model, loader, device, mean, std):
    model.eval()

    all_sample_ids = []
    all_time_ms = []
    all_target_masks = []
    all_preds = []

    for batch in loader:
        sample_id = batch["sample_id"].cpu().numpy()       # [B]
        time_ms = batch["time_ms"].cpu().numpy()           # [B, L]
        x = batch["x"].to(device)
        visible_mask = batch["visible_mask"].to(device)
        observed_input_values = batch["observed_input_values"].to(device)
        target_mask = batch["target_mask"].cpu().numpy()

        pred = model(x, observed_input_values, visible_mask)
        pred = pred.cpu().numpy() * std + mean

        all_sample_ids.append(sample_id)
        all_time_ms.append(time_ms)
        all_target_masks.append(target_mask)
        all_preds.append(pred)

    all_sample_ids = np.concatenate(all_sample_ids, axis=0)      # [N]
    all_time_ms = np.concatenate(all_time_ms, axis=0)            # [N, L]
    all_target_masks = np.concatenate(all_target_masks, axis=0)  # [N, L]
    all_preds = np.concatenate(all_preds, axis=0)                # [N, L]

    return all_sample_ids, all_time_ms, all_target_masks, all_preds


# ============================================================
# MAIN
# ============================================================
print("Device:", cfg.device)

# 1) Load saved model
ckpt = torch.load(cfg.MODEL_PATH, map_location=cfg.device)
hp = ckpt["hp"]
value_mean = float(ckpt["value_mean"])
value_std = float(ckpt["value_std"])

print("Loaded model config:")
print(hp)
print(f"value_mean={value_mean:.6f}, value_std={value_std:.6f}")

model = HybridInpaintingModel(hp).to(cfg.device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

# 2) Load Round 2 test.csv
test_sample_ids, test_times, test_context_mask, test_values = load_signal_csv(
    cfg.TEST_CSV,
    seq_len=cfg.seq_len
)

# 3) Build inference dataset
test_ds = InferenceDataset(
    sample_ids=test_sample_ids,
    times=test_times,
    context_mask=test_context_mask,
    values=test_values,
    mean=value_mean,
    std=value_std,
)

test_loader = DataLoader(
    test_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True,
    drop_last=False,
)

# 4) Predict
pred_sample_ids, pred_time_ms, pred_target_masks, pred_values = predict_dataset(
    model, test_loader, cfg.device, value_mean, value_std
)

print("Shapes before flatten:")
print("pred_sample_ids   :", pred_sample_ids.shape)
print("pred_time_ms      :", pred_time_ms.shape)
print("pred_target_masks :", pred_target_masks.shape)
print("pred_values       :", pred_values.shape)

# 5) Flatten correctly
n_samples, seq_len = pred_values.shape
flat_sample_ids = np.repeat(pred_sample_ids, seq_len)
flat_time_ms = pred_time_ms.reshape(-1)
flat_target_masks = pred_target_masks.reshape(-1)
flat_pred_values = pred_values.reshape(-1)

assert len(flat_sample_ids) == len(flat_time_ms) == len(flat_target_masks) == len(flat_pred_values)

# 6) Build submission
submission = pd.DataFrame({
    "Sample_ID": flat_sample_ids.astype(np.int64),
    "Time_ms": flat_time_ms.astype(np.int64),
    "Predicted_Value": flat_pred_values.astype(np.float32),
})

submission = submission[flat_target_masks > 0.5].copy().reset_index(drop=True)

submission_path = os.path.join(cfg.OUTPUT_DIR, cfg.SUBMISSION_NAME)
submission.to_csv(submission_path, index=False)

print(f"\nSaved submission to: {submission_path}")
print("Submission shape:", submission.shape)
print(submission.head(10))

Device: cuda
Loaded model config:
{'d_model': 192, 'num_conv_blocks': 4, 'num_heads': 4, 'num_decoder_layers': 4, 'ff_mult': 4, 'num_bands': 16, 'dropout': 0.15, 'lr': 0.00015, 'weight_decay': 0.0003, 'batch_size': 128, 'epochs': 80, 'early_stop_patience': 7, 'grad_clip': 1.0, 'mean_val_masked_mse': 0.029541633460919065, 'std_val_masked_mse': 0.0006088687628778629, 'mean_best_epoch': 54}
value_mean=0.500069, value_std=0.248696
Loading CSV: test_features.csv
Original columns: ['Sample_ID', 'Time_ms', 'Is_Context', 'Value']
Cleaned columns : ['Sample_ID', 'Time_ms', 'Is_Context', 'Value']
Loaded 10,000 samples
Observed/context fraction: 0.2000
